# 02. 수치예측과 피크경보 재현

수치예측은 `curve_final`, 최종 피크경보는 `q1_new >= 176`을 사용합니다.
주별 경보기준·주의단계·확률은 비교용이며 최종 경보가 아닙니다.
원본 모델의 학습·예측 방법은 바꾸지 않고 경보 판단만 고정 기준으로 통일합니다.

## 1. 원본 파이프라인 재현

`results/team_submission_code.zip`을 실행용 폴더에 풉니다. 전체 모델 재현에는 시간이 걸립니다. 이미 예측 결과가 있으면 아래 출력 분리 셀만 실행해도 됩니다.

In [ ]:
from pathlib import Path
import os
from zipfile import ZipFile
import nbformat
from nbclient import NotebookClient

ROOT = Path.cwd()
if not (ROOT / 'results/team_submission_code.zip').exists():
    raise FileNotFoundError('real_final 폴더에서 실행하세요.')
PACKAGE = ROOT / 'results/team_submission_code.zip'
RUNTIME = ROOT / 'results/team_submission_runtime'
RUNTIME.mkdir(parents=True, exist_ok=True)
with ZipFile(PACKAGE) as archive:
    root = RUNTIME.resolve()
    for member in archive.namelist():
        if not (root / member).resolve().is_relative_to(root):
            raise ValueError(f'ZIP 경로가 실행 폴더 밖입니다: {member}')
    archive.extractall(root)
source = RUNTIME / 'main_pipeline.ipynb'
team_notebook = nbformat.read(source, as_version=4)
client = NotebookClient(team_notebook, timeout=None, kernel_name=os.environ.get('KAMP_KERNEL', 'python3'), resources={'metadata': {'path': str(RUNTIME)}})
client.execute()
nbformat.write(team_notebook, source)
print('팀원 원본 파이프라인 완료:', source)


## 2. 제출용 출력 분리

원본 예측 파일에는 검증용 실제 전력이 포함됩니다. 이를 제출하지 않고, 기준시각·96칸 정렬을 검사한 뒤 팀원 모델 예측과 경보만 내보냅니다. 파일이 없다면 중단합니다.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
expected = ROOT / 'results/team_submission_runtime/outputs/test_predictions_8_9월.csv'
if not expected.exists():
    expected = ROOT / 'results/model_outputs/test_predictions_8_9월.csv'
df = pd.read_csv(expected).sort_values(['기준시각', '예측칸']).reset_index(drop=True)
if not df.groupby('기준시각')['예측칸'].apply(lambda s: np.array_equal(s.to_numpy(), np.arange(96))).all():
    raise ValueError('각 예측창은 step 0~95를 한 번씩 포함해야 합니다.')
df['최종_경보기준'] = 176.0
df['최종_피크경보'] = (df['q1_new'] >= 176.0).astype(int)
OUT = ROOT / 'results/model_outputs'
OUT.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT / 'test_predictions_8_9월.csv', index=False, encoding='utf-8-sig')
submission = ROOT / 'submission'
submission.mkdir(parents=True, exist_ok=True)
export = df[['기준시각', '예측대상시각', '예측칸', 'curve_final', 'q1_new',
             '최종_경보기준', '최종_피크경보']].copy()
export.to_csv(submission / 'final_predictions_fixed176.csv', index=False, encoding='utf-8-sig')
(OUT / 'selected_alarm_policy.json').write_text(json.dumps({
    'point_model': 'curve_final', 'peak_prediction': 'q1_new',
    'alarm_policy': 'fixed176', 'alarm_threshold': 176.0,
    'weekly_alarm_role': 'comparison_only'
}, ensure_ascii=False, indent=2), encoding='utf-8')
display(export.head())
print('예측창:', export['기준시각'].nunique(), '| 15분 예측:', len(export))
print('제출용 출력에는 실제 전력·주별 경보·확률을 포함하지 않습니다.')


## 다음 단계

03-1~05는 사용자가 만든 LightGBM·GRU 후보의 성능과 시간순 안정성을 확인합니다. 06에서는 **같은 8~9월 예측칸**에 두 접근을 맞춰 비교하고 최종 역할을 확정합니다. 8~9월은 팀원 모델 개발 과정에서 이미 확인했으므로 독립 테스트라고 부르지 않습니다.
